# Homework 09 - Gradient Debugging

目标：训练失败时能定位是哪一类问题。

In [ ]:
from pathlib import Path
import sys, math, random


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True

import importlib
import course.checks as course_checks
importlib.reload(course_checks)
qa_check = course_checks.qa_check

## 完整例子 - grad 累加的症状

如果同一个参数连续 backward，但没有清零，`grad` 会变成累加值。

In [ ]:
from micrograd.engine import Value

w = Value(2.0)
L = w * 3
L.backward()
print('first grad:', w.grad)
L.backward()
print('second grad:', w.grad)

## Modify - 方向反了怎么判断

如果每一步 loss 都稳定变大，先检查是不是写成了 `+ learning_rate * grad`。

In [ ]:
# 填写说明：
# - 完成 student_compare_update_directions，对比错误方向和正确方向。
# - 运行本 cell，看 qa_check 的提示。

def student_compare_update_directions():
    old_w = 5.0
    grad = 8.0
    lr = 0.1
    # TODO: 返回 wrong_update, fixed_update
    pass


qa_check('qa_check_09_modify', globals())


<details><summary>Show / Hide 本题提示</summary>

- 梯度为正，想让 loss 下降，就要让参数往小的方向走。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_compare_update_directions():
    old_w = 5.0
    grad = 8.0
    lr = 0.1
    wrong_update = old_w + lr * grad
    fixed_update = old_w - lr * grad
    return wrong_update, fixed_update
```

</details>


In [ ]:
# 填写说明：
# - 这题拆成四个小实验；最后一个函数调用前面四个。
# - 返回顺序固定：grad_twice_without_zero、wrong_direction_w、fixed_direction_w、dead_relu_grad、unchanged_delta。
# - 运行本 cell，看 qa_check 的提示。

from micrograd.engine import Value


def student_grad_twice_without_zero():
    # TODO: L=3a 连续 backward 两次，return a.grad。
    pass


def student_update_direction_values():
    # TODO: old_w=5, grad=8, lr=0.1，return wrong_update, fixed_update。
    pass


def student_dead_relu_grad():
    # TODO: Value(-2).relu().backward() 后，return 原值的 grad。
    pass


def student_unchanged_param_delta():
    # TODO: 没有执行 update 时，return 参数变化量。
    pass


def student_debug_symptom_probe():
    wrong_direction_w, fixed_direction_w = student_update_direction_values()
    return (
        student_grad_twice_without_zero(),
        wrong_direction_w,
        fixed_direction_w,
        student_dead_relu_grad(),
        student_unchanged_param_delta(),
    )


qa_check('qa_check_debug_mapping', globals())


<details><summary>Show / Hide 本题提示</summary>

- 这题不背症状表，而是用最小实验把症状跑出来。
- dead ReLU 可以用 `Value(-2.0).relu()` 观察。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def student_grad_twice_without_zero():
    a = Value(2.0)
    L = a * 3
    L.backward()
    L.backward()
    return a.grad


def student_update_direction_values():
    old_w = 5.0
    grad = 8.0
    lr = 0.1
    wrong_direction_w = old_w + lr * grad
    fixed_direction_w = old_w - lr * grad
    return wrong_direction_w, fixed_direction_w


def student_dead_relu_grad():
    r = Value(-2.0)
    out = r.relu()
    out.backward()
    return r.grad


def student_unchanged_param_delta():
    before = 5.0
    after_without_update = 5.0
    return after_without_update - before


def student_debug_symptom_probe():
    wrong_direction_w, fixed_direction_w = student_update_direction_values()
    return (
        student_grad_twice_without_zero(),
        wrong_direction_w,
        fixed_direction_w,
        student_dead_relu_grad(),
        student_unchanged_param_delta(),
    )
```

</details>


In [ ]:
# 填写说明：
# - 修复 broken_update：不要返回代码文字，直接让函数真的更新参数。
# - 运行本 cell，看 qa_check 的提示。

# Debug Lab：下面这个更新方向是错的。
def broken_update(p, lr):
    p.data = p.data + lr * p.grad


def fixed_update(p, lr):
    # TODO: 用正确方向修改 p.data。
    pass


qa_check('qa_check_fixed_update_line', globals(), fixed_update)


<details><summary>Show / Hide 本题提示</summary>

- 这次不要写文字答案，而是让函数真的把 `p.data` 改对。

</details>


<details><summary>Show / Hide 本题答案</summary>

```python
def fixed_update(p, lr):
    p.data = p.data - lr * p.grad
```

</details>


<details><summary>Show / Hide 提示</summary>

先找完整例子的形状，再只改一个地方。填 TODO 前，先用小数字在纸上算一遍；测试失败时先判断错在数学、Python、计算图还是训练循环。

</details>

<details><summary>Show / Hide 答案</summary>

答案不要一开始看。正确答案应该能由完整例子和 Modify 题一步推出；如果你需要直接看答案，说明前一个台阶还没踩稳。

</details>

## 提交清单

- [ ] 我跑过完整例子。
- [ ] 我完成了 Modify 题。
- [ ] 我完成了 TODO 作业并通过 qa_check。
- [ ] 我修过 Debug Lab。
- [ ] 我能说出本节知识如何接回 micrograd 主线。